
Các bước:
1. Import thư viện
2. Đọc dữ liệu từ `data/split.csv`
3. Xây dựng mô hình
4. Huấn luyện
5. Vẽ biểu đồ học
6. Đánh giá trên tập test
7. Lưu kết quả

## Bước 1 – Import thư viện

In [2]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

# Tìm thư mục gốc của dự án (nơi có config.py), không phụ thuộc thư mục chạy notebook.
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục dự án chứa config.py")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config

ROOT = PROJECT_ROOT
RESULTS_DIR = ROOT / "results" / "custom_cnn"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
(ROOT / "models").mkdir(parents=True, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("split.csv exists:", config.SPLIT_CSV_PATH.exists())
print("CPUs available:", tf.config.list_physical_devices("CPU"))

TensorFlow: 2.21.0
PROJECT_ROOT: d:\Fire-and-smoke-detection-using-D-Fire
split.csv exists: True
CPUs available: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## Bước 2 – Đọc dữ liệu
`split.csv` chứa đường dẫn tới 4800 ảnh đã resize 224x224 và cột `split` (train/val/test).

In [4]:
df = pd.read_csv(config.SPLIT_CSV_PATH)

print("Số hàng:", len(df))
print(df.groupby(["split", "label"]).size().unstack())

Số hàng: 4800
label  fire  nofire  smoke  smokefire
split                                
test    200     200    200        200
train   800     800    800        800
val     200     200    200        200


In [5]:
# Đọc 1 ảnh từ đường dẫn, chuẩn hoá về [0, 1]
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32) / 255.0
    img.set_shape([224, 224, 3])
    return img, label


# Tạo dataset cho từng tập (train / val / test)
def make_dataset(split_name, shuffle=False):
    part = df[df["split"] == split_name].reset_index(drop=True)
    paths = [str(ROOT / p) for p in part["processed_filepath"]]
    labels = part["class_index"].astype(int).tolist()

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        ds = ds.shuffle(len(paths), seed=config.RANDOM_SEED)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(32).prefetch(tf.data.AUTOTUNE)
    return ds, part


train_ds, train_df = make_dataset("train", shuffle=True)
val_ds, val_df = make_dataset("val")
test_ds, test_df = make_dataset("test")

print("Train:", len(train_df), "| Val:", len(val_df), "| Test:", len(test_df))

Train: 3200 | Val: 800 | Test: 800


## Bước 3 – Xây dựng mô hình

CNN chỉ xếp Conv2D liên tiếp thì khó học khi mạng sâu (gradient biến mất) và không biết kênh nào quan trọng. Nên mình dùng:
- **Residual (đường tắt)**: cộng input vào output để gradient đi thẳng được.
- **SE block**: mạng tự học trọng số cho từng kênh (lửa là kênh cam/đỏ, khói là vùng mờ).
- **Global Average Pooling**: thay Flatten để ít tham số hơn, đỡ overfit.

In [6]:
# Khối residual có kèm SE
def res_block(x, filters, stride=1):
    shortcut = x

    # 2 lớp conv
    x = layers.Conv2D(filters, 3, strides=stride, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(filters, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)

    # SE: mỗi kênh gom thành 1 số, học trọng số 0-1 rồi nhân lại
    se = layers.GlobalAveragePooling2D()(x)
    se = layers.Dense(filters // 16, activation="relu")(se)
    se = layers.Dense(filters, activation="sigmoid")(se)
    se = layers.Reshape((1, 1, filters))(se)
    x = layers.Multiply()([x, se])

    # nếu kích thước đổi thì shortcut cũng phải đổi theo mới cộng được
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride)(shortcut)

    x = layers.Add()([x, shortcut])
    return layers.Activation("relu")(x)

In [8]:
inputs = layers.Input(shape=(224, 224, 3))

# conv đầu, stride 2 giảm ảnh còn 112x112
x = layers.Conv2D(32, 3, strides=2, padding="same", activation="relu")(inputs)

x = res_block(x, 64)                # 112x112
x = res_block(x, 128, stride=2)     # 56x56
x = res_block(x, 128)
x = res_block(x, 256, stride=2)     # 28x28
x = res_block(x, 256)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation="relu")(x)
outputs = layers.Dense(len(config.CLASS_NAMES), activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_14 (Conv2D)  │ (None, 112, 112,  │        896 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 112, 112,  │     18,496 │ conv2d_14[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_15[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_10       │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 112, 112,  │     36,928 │ activation_10[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_16[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 4)         │        260 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 64)        │        320 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_5 (Reshape) │ (None, 1, 1, 64)  │          0 │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_5          │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Multiply)          │ 64)               │            │ reshape_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 112, 112,  │      2,112 │ conv2d_14[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 112, 112,  │          0 │ multiply_5[0][0], │
│                     │ 64)               │            │ conv2d_17[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_11       │ (None, 112, 112,  │          0 │ add_5[0][0]       │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 56, 56,    │     73,856 │ activation_11[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ conv2d_18[0][0]   │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_12       │ (None, 56, 56,    │          0 │ batch_normalizat

 Total params: 2,743,736 (10.47 MB)

 Trainable params: 2,740,408 (10.45 MB)

 Non-trainable params: 3,328 (13.00 KB)

## Bước 4 – Compile và huấn luyện
Chạy tối đa 50 epoch, `EarlyStopping` sẽ dừng sớm nếu val_loss không giảm nữa và lấy lại trọng số tốt nhất.

In [9]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              metrics=["accuracy"])

callbacks = [
    # dừng nếu val_loss không giảm sau 10 epoch
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=10,
                                     restore_best_weights=True),
    # giảm learning rate một nửa nếu val_loss đứng yên 4 epoch
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                         patience=4, min_lr=1e-6),
]

In [10]:
H = model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks)

Epoch 1/50


c:\Users\OWNER\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


KeyboardInterrupt: 

In [ ]:
model.save(ROOT / "models" / "custom_cnn.keras")

## Bước 5 – Biểu đồ quá trình học

In [ ]:
plt.figure(figsize=(12, 4))
for i, name in enumerate(["accuracy", "loss"]):
    plt.subplot(1, 2, i + 1)
    plt.plot(H.history[name], label="train")
    plt.plot(H.history["val_" + name], label="val")
    plt.title(name)
    plt.xlabel("epoch")
    plt.legend()

plt.savefig(RESULTS_DIR / "learning_curves.png", dpi=150)
plt.show()

## Bước 6 – Đánh giá trên tập test

In [ ]:
# model đã được EarlyStopping khôi phục về trọng số tốt nhất nên dùng luôn
y_prob = model.predict(test_ds)
y_pred = np.argmax(y_prob, axis=1)
y_true = test_df["class_index"].astype(int).values

print("Đoán đúng:", (y_pred == y_true).sum(), "/", len(y_true))
print(classification_report(y_true, y_pred, target_names=config.CLASS_NAMES, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=config.CLASS_NAMES, yticklabels=config.CLASS_NAMES)
plt.xlabel("Dự đoán")
plt.ylabel("Thực tế")
plt.title("Confusion matrix")
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150)
plt.show()

## Bước 7 – Lưu kết quả

In [ ]:
metrics = {
    "model_name": "Custom CNN (Residual + SE)",
    "test_accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
    "test_macro_f1": round(float(f1_score(y_true, y_pred, average="macro")), 4),
    "total_parameters": int(model.count_params()),
    "epochs_trained": len(H.history["loss"]),
}

with open(RESULTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print(metrics)